In [2]:
import pandas as pd
import numpy as np

def answer_one():
    energy = pd.read_excel('D:/Energy Indicators.xls', skiprows=17, skipfooter=38)
    energy = energy.iloc[:, 2:]
    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']
    energy.replace('...', np.nan, inplace=True)
    energy['Energy Supply'] *= 1000000
    
    def clean_country(data):
        data = ''.join([i for i in data if not i.isdigit()])
        if '(' in data: data = data.split('(')[0]
        return data.strip()
    
    energy['Country'] = energy['Country'].apply(clean_country)
    energy['Country'] = energy['Country'].replace({
        "Republic of Korea": "South Korea",
        "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    })

    gdp_raw = pd.read_excel('D:/world_bank.xls', header=None)
    start_row = 0
    for i, row in gdp_raw.iterrows():
        if "Country Name" in row.values:
            start_row = i
            break
    gdp = pd.read_excel('D:/world_bank.xls', skiprows=start_row)
    
    c_col = gdp.columns[0]
    gdp[c_col] = gdp[c_col].astype(str).str.strip()
    gdp[c_col] = gdp[c_col].replace({
        "Korea, Rep.": "South Korea", 
        "Iran, Islamic Rep.": "Iran", 
        "Hong Kong SAR, China": "Hong Kong"
    })
    
    gdp.columns = [str(c).strip().split('.')[0] for c in gdp.columns]
    years = [str(y) for y in range(2006, 2016)]
    GDP = gdp[['Country' if 'Country' in gdp.columns else gdp.columns[0]] + years].copy()
    GDP.columns = ['Country'] + years

    scim = pd.read_excel('D:/scimagojr-3.xlsx')
    scim_top15 = scim[:15]

    df = pd.merge(scim_top15, energy, how='inner', on='Country')
    df = pd.merge(df, GDP, how='inner', on='Country')
    return df.set_index('Country')

def answer_two():
    Top15 = answer_one()
    years = [str(y) for y in range(2006, 2016)]
    for y in years: 
        Top15[y] = pd.to_numeric(Top15[y], errors='coerce')
    return Top15[years].mean(axis=1).sort_values(ascending=False)

def answer_three():
    Top15 = answer_one()
    avgGDP = answer_two()
    country = avgGDP.index[5]
    return Top15.loc[country, '2015'] - Top15.loc[country, '2006']

def answer_four():
    Top15 = answer_one()
    Top15['Ratio'] = Top15['Self-citations'] / Top15['Citations']
    return (Top15['Ratio'].idxmax(), Top15['Ratio'].max())

def answer_five():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    return Top15['PopEst'].sort_values(ascending=False).index[2]

def answer_six():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['PopEst']
    return Top15['Citable docs per Capita'].astype(float).corr(Top15['Energy Supply per Capita'].astype(float))

def answer_seven():
    Top15 = answer_one()
    ContinentDict  = {'China':'Asia', 'United States':'North America', 'Japan':'Asia', 
                      'United Kingdom':'Europe', 'Russian Federation':'Europe', 
                      'Canada':'North America', 'Germany':'Europe', 'India':'Asia',
                      'France':'Europe', 'South Korea':'Asia', 'Italy':'Europe', 
                      'Spain':'Europe', 'Iran':'Asia', 'Australia':'Australia', 
                      'Brazil':'South America'}
    Top15['PopEst'] = (Top15['Energy Supply'] / Top15['Energy Supply per Capita']).astype(float)
    Top15['Continent'] = [ContinentDict[country] for country in Top15.index]

    return Top15.groupby('Continent')['PopEst'].agg(['size', 'sum', 'mean', 'std'])

if __name__ == "__main__":
    try:
        print("--- Завдання 1: Об'єднана таблиця (перші 3 рядки) ---")
        print(answer_one().head(3))
        
        print("\n--- Завдання 2: Середній ВВП ---")
        print(answer_two())
        
        print("\n--- Завдання 3: Зміна ВВП для 6-ї країни ---")
        print(answer_three())
        
        print("\n--- Завдання 4: Макс. співвідношення самоцитувань ---")
        print(answer_four())
        
        print("\n--- Завдання 5: 3-тя країна за населенням ---")
        print(answer_five())
        
        print("\n--- Завдання 6: Кореляція ---")
        print(answer_six())
        
        print("\n--- Завдання 7: Статистика по континентах ---")
        print(answer_seven())
    except Exception as e:
        print(f"Помилка при виконанні: {e}")

--- Завдання 1: Об'єднана таблиця (перші 3 рядки) ---
               Rank            Region  Documents  Citable documents  \
Country                                                               
China             1    Asiatic Region     273437             272374   
United States     2  Northern America     175891             172431   
India             3    Asiatic Region      55082              53775   

               Citations  Self-citations  Citations per document  H index  \
Country                                                                     
China            2336764         1615239                    8.55      245   
United States    2230544          724472                   12.68      363   
India             463165          162944                    8.41      181   

              Energy Supply Energy Supply per Capita  ...          2006  \
Country                                               ...                 
China          127191000000                       93  